# Solution Template

Use this notebook as a guide to implement your solution. Keep in mind that some cells should remain as they are so that you code works properly, for instance, the following cell in which the required libraries are imported.

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx # for drawing graphs
import matplotlib.pyplot as plt # for drawing graphs
from pybbn.graph.dag import Bbn # for creating Bayesian Belief Networks (BBN)
from pybbn.graph.edge import Edge, EdgeType
from pybbn.graph.jointree import EvidenceBuilder
from pybbn.graph.node import BbnNode
from pybbn.graph.variable import Variable
from pybbn.pptc.inferencecontroller import InferenceController

Just run the next cell to load the data.

In [ ]:
diabetes = pd.read_csv('diabetes-dataset.csv')
diabetes.head()

Create a new column called `Overweight` in which a person whose `BMI` is above 25 will be tagged as a one, and zero otherwise.

In [ ]:
# Derive Overweight from the ORIGINAL BMI, before discretizing.
# BMI >= 25 is overweight; BMI <= 0 denotes a missing measurement.
diabetes['Overweight'] = np.where(
    diabetes['BMI'] <= 0,
    np.nan,
    np.where(diabetes['BMI'] >= 25, 1, 0)
)
diabetes[['BMI', 'Overweight']].head()

You are to code the next function, which discretize all the variables of the dataset, except `Outcome` and `Overweight`. Remember that you will discretize with respect to the quantiles of each variables: if a variable value is less than Q1, then said value is replaced by a **zero**; if the given value is greater or equal than Q1 but less than Q2, then the value should be replaced by a **one**; if the variable value is greater or equal than Q2 but less than Q3, then the value should be replaced by a **two**; finally, if a variable value is greater than Q3, it should be assigned the value **three**. 

In [ ]:
def discretize(df):
    """
    Discretize every variable except Outcome and Overweight into quartiles.

    Medical measurements equal to zero are treated as missing, but zero
    pregnancies is a valid observation and is preserved.
    Values at a quartile boundary belong to the higher category.
    """
    discretized_df = df.copy()

    for column in discretized_df.columns:
        if column in ('Outcome', 'Overweight'):
            continue

        original = discretized_df[column]

        # For clinical measurements, zero represents an unavailable value.
        # For Pregnancies, however, zero is a legitimate count.
        if column == 'Pregnancies':
            valid_values = original.dropna()
            usable = original
        else:
            valid_values = original[original > 0]
            usable = original.where(original > 0, np.nan)

        q1, q2, q3 = valid_values.quantile([0.25, 0.50, 0.75])

        discretized_df[column] = np.select(
            [usable < q1,
             (usable >= q1) & (usable < q2),
             (usable >= q2) & (usable < q3),
             usable >= q3],
            [0, 1, 2, 3],
            default=np.nan
        )

    return discretized_df

In [ ]:
discrete_df = discretize(diabetes)
discrete_df

In the following cel you are to create two dictionaries: `graph` will store the topology of the Bayesian network, so each element is associated to a list that contains the names of the parents of said element; `values` stores the values that each variable of the network takes, which are the discrete values that were computed above.

In [ ]:
# Parent lists follow the diabetes network from Bayesian Networks.ipynb.
# In this template we keep the ORIGINAL column names.
graph = {
    'Overweight': [],
    'DiabetesPedigreeFunction': [],
    'Age': [],
    'Pregnancies': [],
    'SkinThickness': ['Overweight'],
    'BMI': ['Overweight'],
    'Outcome': ['Age', 'DiabetesPedigreeFunction', 'Pregnancies', 'Overweight'],
    'BloodPressure': ['Outcome', 'Overweight'],
    'Insulin': ['Outcome'],
    'Glucose': ['Outcome']
}

# Ordered discrete states of every node, excluding missing observations.
values = {
    node: sorted(int(value) for value in discrete_df[node].dropna().unique())
    for node in graph
}

graph, values

The next function obtains the probabilities of a given node. This function will be used later to create a dictionary in which each element contains a node and its list of probabilities.

In [ ]:
from itertools import product

def probabilities(df, node):
    """
    Return a flattened CPT for the given node, in the order of its
    parent combinations followed by the node's ordered values.
    Uses only the relevant, nonmissing columns and Laplace smoothing.
    """
    parents = graph[node]
    node_states = values[node]
    data = df[parents + [node]].dropna()
    alpha = 1
    probabilities_list = []

    combinations = (
        product(*(values[parent] for parent in parents))
        if parents else [()]
    )

    for combination in combinations:
        subset = data
        for parent, state in zip(parents, combination):
            subset = subset[subset[parent] == state]

        counts = subset[node].value_counts()
        denominator = len(subset) + alpha * len(node_states)
        probabilities_list.extend([
            float((counts.get(state, 0) + alpha) / denominator)
            for state in node_states
        ])

    return probabilities_list

The following function must create a dictionary in which item is a node and its corresponding list of probabilities

In [ ]:
def tables(df):
    """Compute the marginal and conditional tables for all graph nodes."""
    probabilities_tables = {}

    for node in graph:
        probabilities_tables[node] = probabilities(df, node)

    return probabilities_tables

# Compute CPTs once, so they can be used when creating the nodes.
probabilities_tables = tables(discrete_df)
{node: len(table) for node, table in probabilities_tables.items()}

### Validate the probability tables

As in our Bayesian Networks notebook, each marginal or conditional probability distribution must sum to 1. The table checks all rows, including combinations of parent states that were not observed in the data.

In [ ]:
def validate_tables(network, states, all_tables, tolerance=1e-9):
    results = []
    for node in network:
        block_size = len(states[node])
        table = np.asarray(all_tables[node], dtype=float)
        expected_blocks = int(np.prod([len(states[p]) for p in network[node]])) if network[node] else 1
        assert table.size == expected_blocks * block_size, f'Invalid CPT length: {node}'
        sums = table.reshape(-1, block_size).sum(axis=1)
        valid = bool(np.all(np.isfinite(table)) and
                     np.all(table >= 0) and
                     np.allclose(sums, 1.0, atol=tolerance))
        results.append({'Node': node, 'Distributions': len(sums),
                        'Minimum sum': float(sums.min()),
                        'Maximum sum': float(sums.max()),
                        'Valid': valid})
    return pd.DataFrame(results)

cpt_validation = validate_tables(graph, values, probabilities_tables)
assert cpt_validation['Valid'].all(), 'At least one CPT is invalid.'
cpt_validation

Create the nodes of the network in this cell. For each line, replace `"node index"` and the empty list by the proper variable name and variable values, respectively.

In [ ]:
# Instantiate every variable with its node ID, states, and probability table.
overweight = BbnNode(
    Variable(0, 'Overweight', [str(v) for v in values['Overweight']]),
    probabilities_tables['Overweight']
)
diabetes_pedigree_function = BbnNode(
    Variable(1, 'DiabetesPedigreeFunction', [str(v) for v in values['DiabetesPedigreeFunction']]),
    probabilities_tables['DiabetesPedigreeFunction']
)
age = BbnNode(
    Variable(2, 'Age', [str(v) for v in values['Age']]),
    probabilities_tables['Age']
)
pregnancies = BbnNode(
    Variable(3, 'Pregnancies', [str(v) for v in values['Pregnancies']]),
    probabilities_tables['Pregnancies']
)
skin_thickness = BbnNode(
    Variable(4, 'SkinThickness', [str(v) for v in values['SkinThickness']]),
    probabilities_tables['SkinThickness']
)
bmi = BbnNode(
    Variable(5, 'BMI', [str(v) for v in values['BMI']]),
    probabilities_tables['BMI']
)
outcome = BbnNode(
    Variable(6, 'Outcome', [str(v) for v in values['Outcome']]),
    probabilities_tables['Outcome']
)
blood_pressure = BbnNode(
    Variable(7, 'BloodPressure', [str(v) for v in values['BloodPressure']]),
    probabilities_tables['BloodPressure']
)
insulin = BbnNode(
    Variable(8, 'Insulin', [str(v) for v in values['Insulin']]),
    probabilities_tables['Insulin']
)
glucose = BbnNode(
    Variable(9, 'Glucose', [str(v) for v in values['Glucose']]),
    probabilities_tables['Glucose']
)

Implement your graph in the following cell. Add as many nodes and edges as necessary. Replace the strings by the proper variables.

In [ ]:
# Create nodes and edges using the parent dictionary above.
bbn_nodes = {
    'Overweight': overweight,
    'DiabetesPedigreeFunction': diabetes_pedigree_function,
    'Age': age,
    'Pregnancies': pregnancies,
    'SkinThickness': skin_thickness,
    'BMI': bmi,
    'Outcome': outcome,
    'BloodPressure': blood_pressure,
    'Insulin': insulin,
    'Glucose': glucose
}

bbn = Bbn()
for node in bbn_nodes.values():
    bbn.add_node(node)

for child, parents in graph.items():
    for parent in parents:
        bbn.add_edge(Edge(bbn_nodes[parent], bbn_nodes[child], EdgeType.DIRECTED))

Do not forget to run this cell and do not modify it, inferences depend on it.

In [ ]:
# Convert the BBN to a join tree. Do not modify this cell.

join_tree = InferenceController.apply(bbn)

The following cell is very useful for visualizing your Bayesian network. It is very recommended that you make the necessary changes and run it to verify that your network was implementented correctly.

In [ ]:
# Positions keyed by node IDs (matching the BbnNode Variable IDs).
pos = {
    0: (3.4, 1.5),    # Overweight
    1: (-1.7, 1.5),   # DiabetesPedigreeFunction
    2: (-3.9, 1.5),   # Age
    3: (0.3, 1.5),    # Pregnancies
    4: (2.6, -1.5),   # SkinThickness
    5: (4.2, -1.5),   # BMI
    6: (0.0, 0.0),    # Outcome
    7: (0.9, -1.5),   # BloodPressure
    8: (-2.5, -1.5),  # Insulin
    9: (-0.9, -1.5)   # Glucose
}

options = {
    'font_size': 9,
    'node_size': 6500,
    'node_color': 'yellow',
    'edgecolors': 'black',
    'edge_color': 'red',
    'linewidths': 2,
    'width': 2.2
}

n, d = bbn.to_nx_graph()
plt.figure(figsize=(16, 7))
nx.draw(n, with_labels=True, labels=d, pos=pos, **options)
ax = plt.gca()
ax.margins(0.25)
plt.axis('off')
plt.show()

The goal of `print_probs` is to print out the probability distributions of all the nodes of the network. You can modify this code to print only the distributions of certain nodes if you find that helpful.

In [ ]:
# Define a function for printing marginal probabilities.

def print_probs():
    for node in join_tree.get_bbn_nodes():
        potential = join_tree.get_bbn_potential(node)
        print("Node:", node)
        print("Values:")
        print(potential)
        print('----------------')
    
# Use the above function to print marginal probabilities.

print_probs()

The function `evidence` helps tyou to create evidence that will be used for making inferences. Do not modify this cell, please.

In [ ]:
# To add evidence of events that happened so probability distribution can be recalculated.

def evidence(ev, nod, val, like):
    ev = EvidenceBuilder() \
    .with_node(join_tree.get_bbn_node_by_name(nod)) \
    .with_evidence(val, like) \
    .build()
    join_tree.set_observation(ev)

Now you are ready to add evidence and print out the new distributions of your network. 

In [ ]:
# Example: observe a patient in the highest Age and Glucose quartiles.
# The helper uses string labels, which match the node states.
join_tree = InferenceController.apply(bbn)
evidence('ev1', 'Age', '3', 1)
evidence('ev2', 'Glucose', '3', 1)

# Posterior distributions after adding evidence.
print_probs()

If you need to reset the Bayesian network, rerun this line of code or rerun the above cell twice.

In [ ]:
join_tree = InferenceController.apply(bbn)

### Compare three hypothetical patients

After resetting the join tree, compare three hypothetical profiles using the same evidence-updating method. Each profile uses a **new** join tree, so observations from one patient do not affect another. These scenarios demonstrate inference; they are not clinical diagnoses.

In [ ]:
def evaluate_patient_profile(profile):
    patient_join_tree = InferenceController.apply(bbn)

    for node_name, state in profile.items():
        ev = (EvidenceBuilder()
              .with_node(patient_join_tree.get_bbn_node_by_name(node_name))
              .with_evidence(str(state), 1.0)
              .build())
        patient_join_tree.set_observation(ev)

    return patient_join_tree.get_posteriors()['Outcome']


patient_profiles = {
    'Low-risk profile': {
        'Age': 0, 'DiabetesPedigreeFunction': 0,
        'Glucose': 0, 'BMI': 0, 'Overweight': 0
    },
    'Intermediate-risk profile': {
        'Age': 2, 'DiabetesPedigreeFunction': 2,
        'Glucose': 2, 'BMI': 2, 'Overweight': 1
    },
    'High-risk profile': {
        'Age': 3, 'DiabetesPedigreeFunction': 3,
        'Glucose': 3, 'BMI': 3, 'Overweight': 1
    }
}

prior_diabetes_probability = InferenceController.apply(bbn).get_posteriors()['Outcome']
print(f"Prior P(Outcome = 1): {prior_diabetes_probability['1']:.2%}")

patient_results = []
for name, profile in patient_profiles.items():
    posterior = evaluate_patient_profile(profile)
    patient_results.append({
        'Profile': name,
        'P(Outcome=0)': float(posterior['0']),
        'P(Outcome=1)': float(posterior['1'])
    })

patient_results = pd.DataFrame(patient_results)
patient_results

The table compares the posterior probability of `Outcome = 1` (diabetes) after observing the specified patient variables. The probabilities describe this fitted Bayesian network only; the model is not validated for medical diagnosis.